# Environment Setup & Model Loading

This notebook covers the environment and model-loading stage and ends with a smoke test that confirms the inference path works.

**Colab GPU:** Prefer L4 or A100


## 1. Check the GPU 


In [1]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  sm_{p.major}{p.minor}  {p.total_memory/1024**3:.1f} GB')
print('FlashAttention-2 / native bf16 available:' , (p.major, p.minor) >= (8, 0))

GPU: NVIDIA A100-SXM4-80GB  sm_80  79.3 GB
FlashAttention-2 / native bf16 available: True


## 2. Mount Google Drive

Caching the HF weights in Drive.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF cache ->', os.environ['HF_HOME'])

Mounted at /content/drive
HF cache -> /content/drive/MyDrive/openvla_cache/hf


## 3. Install Pinned Dependencies

> Do **not** reinstall torch, Colab's build is matched to its CUDA driver.


In [3]:
 !pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 \
    huggingface_hub==0.23.4 accelerate==0.30.1 'bitsandbytes>=0.45.0' \
    -q --upgrade "protobuf>=6.31.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 12.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 159.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 33.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4

**After this install, restart the runtime once** because Colab pre-imports a newer transformers. Then re-run cells 1-2 and skip this install cell.


## 4. Get the code and import from `model.py`

Clones the repo fresh each session and imports the loader/inference/logging functions.

In [4]:
import sys, shutil, os
REPO_DIR = '/content/ECS8056'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)   # remove stale copy so the latest push is used
!git clone -q https://github.com/LewisTL/ECS8056.git {REPO_DIR}
sys.path.append(REPO_DIR)     # adjust if model.py lives in a subfolder of the repo
from model import load_openvla, predict_action, run_metadata, append_prediction_log
print('Imported from', REPO_DIR + '/model.py')

Imported from /content/ECS8056/model.py


## 5. Load OpenVLA-7B

In [5]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)   # GPU, dtype, seed, library versions for logging
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-80GB (sm_80, 79.3 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-80GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.0'}


## 6. Smoke Test

A synthetic frame is sufficient here; the only check is that the pipeline produces
a well-formed 7-DoF action `[dx, dy, dz, droll, dpitch, dyaw, gripper]`. Real
BridgeData V2 frames come in the next notebook.


In [6]:
import numpy as np
from PIL import Image

CSV_PATH = '/content/drive/MyDrive/openvla_cache/predictions.csv'
dummy = Image.fromarray((np.random.default_rng(0).random((224,224,3))*255).astype(np.uint8))

instr = 'pick up the object on the left'
action = predict_action(processor, vla, dummy, instr, compute_dtype)
append_prediction_log(CSV_PATH, action, instr, meta, scene_id='smoke', spatial_term='left')

print('Action shape :', action.shape)
print('Action vector:', np.round(action, 4))
assert action.shape == (7,), 'Expected a 7-DoF vector'


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (4304) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Action shape : (7,)
Action vector: [-0.0029  0.0161 -0.006  -0.0014  0.0086  0.203   0.    ]


## 6. Sanity Check

Same image, two instructions differing only in the spatial term. With a random
image, a clean sign flip should not be expected. This cell just confirms the
probe mechanics (two calls, compare `dx`) work before real scenes are wired in.


In [7]:
left  = predict_action(processor, vla, dummy, 'move to the cup on the left',  compute_dtype)
right = predict_action(processor, vla, dummy, 'move to the cup on the right', compute_dtype)
print('dx(left) = %.4f   dx(right) = %.4f   diff = %.4f' % (left[0], right[0], left[0]-right[0]))

dx(left) = -0.0078   dx(right) = 0.0031   diff = -0.0110
